# Context for Think, Observe and the Rest

**Prerequisites:** `04_context_construction.ipynb`. From `02_agent_runtime`: `02_the_agent_loop.ipynb`

**The runtime decides *when* a component runs. Context engineering decides *what it sees*. Chapter 2 has nowhere to say the second thing.**

So every component says it for itself, inside its own `run()`:

| Component | What it gives the model | Decided in |
|---|---|---|
| `Think` | the whole conversation, rebuilt in the API's native tool-call shape | `Think._build_messages()` |
| `Observe` | the whole conversation flattened to text, plus the raw tool result | `Observe.run()` |
| `Graph` | the whole conversation, flattened, plus the raw result | `Graph._phrase()` |
| `ReflectionThink`, `SelectThink` | the whole conversation, flattened | each one's `run()` |

Four context decisions, four places, all of them "everything". None of them visible from the loop where the components are declared:

```python
Runtime(loop=[Think(tools=tools), Act(functions=functions), Observe()], repeat_from=0)
```

Nothing there says what any of them gets. This notebook makes that a thing you write, and it changes no chapter 2 code to do it. Two components, two very different answers: `Think` gets a scored selection of the run, and `Observe` gets only the pass it is reporting on.

## Setup

## What each component is given today

`Think` hands back its messages without calling anything. For `Observe`, the module's `chat` is swapped for a recorder, so nothing is sent. Same trick as notebook 1.

`Observe`'s job is one sentence: take what a tool returned and say what it says. For that it is handed the entire session — the rules, the plan, the search-api detour, every earlier reading — and then asked to describe one number.

It is not wasted only in tokens. Chapter 2 had to write two separate warnings into that prompt because of what the conversation does to it: *"Copy every value exactly as the tool gave it, in the units the tool gave it. Do not convert, calculate, round, combine or reformat anything — not even when the conversation asks for a different unit."* That instruction exists because the conversation was asking for a different unit, and `Observe` obliged: 20 °C came back as 68 °F, converted by a component whose whole purpose is to not do that.

**A component that cannot see the request cannot be tempted by it.**

## What is context, and what is machinery

A component reads a lot off the conversation. Almost none of it is context:

| Attribute | What it is |
|---|---|
| `messages` | **context** — what the model gets shown |
| `pending`, `step_outputs` | the result of the last act, and every raw result so far |
| `branches`, `running`, `abandoned`, `pool`, `frontier` | sub-agent machinery |
| `remaining_budget`, `reserved_budget` | budgets |

Only the first is worth narrowing, and the rest must arrive untouched. So `View` overrides exactly one attribute and delegates the rest — reads *and* writes, because a component setting `conversation.pending` has to set it on the conversation the runtime is carrying, not on a copy that is about to be thrown away.

## Items back into a conversation

A component expects a conversation, so the selected items have to go back into the runtime's own message shapes — not the API's. That matters for `Think`: it rebuilds native tool calls from exactly the `decision` + `tool` pairing, and chapter 2 found models recognise finished work from that structure and not from prose about it.

An `observation` item still carries the call it came from in `about`, so the pair can be rebuilt. Items that were never messages — facts, summaries, scratchpad state — have no shape to go back to, and become one labelled block rather than turns somebody took.

## `Focus`: a component and the context it gets

`Focus` wraps a component. It works out which items that component should get, rebuilds them into a `View`, and calls the component with that. The component is unchanged and unaware; `Runtime` is unchanged and unaware, because a `Focus` is itself just something with `.run(conversation)`.

For `Observe` the right policy is not a scored selection at all. `Observe` phrases the result of the call that just happened, and that result arrives in `conversation.pending`, never through `messages`. So what it needs from the conversation is **this pass of the loop**: the standing request, so it names the entity correctly, and anything that came back since the last decision — which in a live loop is nothing, because `Observe` runs immediately after `Act`.

`current_pass` is that boundary, and it drops two things on purpose:

- **the rules, even though they are pinned.** Pinned means "must reach the component that answers the user", not "every component". Notebook 2 pinned them so nothing could lose them; here, losing them is the goal.
- **everything older** — the plan, the agent's own past answers, earlier readings.
- **the raw `tool_result` item**, because that same JSON already reaches `Observe` through `pending`. Keeping the item too put it in the prompt twice: 146 tokens of pure duplication on this session's metrics call.

What it does keep, besides the request, is the **decision itself** — the call `Think` just made, arguments included. That needed a change to `available()`: a `tool_call` decision is normally fused with the tool message after it into one `observation`, and mid-loop that message does not exist yet, because it is what `Observe` is about to produce. So the call in flight was in no item at all, and the one component whose job is to describe it could not see which call it was. It now arrives as a `decision` item.

A scored policy cannot say any of this. Given weights it will always trade part of this pass for something older that scored well.

Two items, and neither is a rule: 851 tokens down to 398. It did not fall further because `Observe`'s message is three things: its own instructions (~180 tokens, fixed), the raw tool result (~150, which no context policy touches — trimming that is `invoke(max_tokens=)` in chapter 3 or `truncate()` in notebook 3), and the conversation. Only the third is ours to cut, and it went from ~660 tokens to ~40.

Note what `current_pass` keeps here: the observation from turn 19, because this canned session carries on for two more turns after that decision. Mid-loop — which is when `Observe` actually runs — the shape is different, and simpler:

Does it still do its job?

Both sentences carry the same values off the same JSON body — which is the point: the body is in `pending`, and a narrowed view leaves it alone. Nothing was lost by not showing `Observe` the rules, the plan, the search-api detour or the earlier readings.

That is the easy half of the case: the same result for 398 tokens instead of 851, on a component that runs once per tool call for the whole life of the agent. Mid-loop, as the cell above shows, it is smaller still.

## What a narrowed context replaces

The clearest argument for giving `Observe` less is not the tokens. It is what chapter 2 had to write into `Observe`'s prompt to survive seeing the whole conversation:

> *"Copy every value exactly as the tool gave it, in the units the tool gave it. Do not convert, calculate, round, combine or reformat anything — not even when the conversation asks for a different unit."*

That paragraph exists because the conversation was asking for a different unit and `Observe` obliged: 20 °C came back as 68 °F, arithmetic no tool did, reaching the conversation as if a sensor had reported it.

The prompt fix works. Run the shipped `Observe` on a result reported in **seconds**, with the milliseconds rule in context and without it, and it converts nothing: 8 runs, 8 refusals, both ways. So the guard is earning its keep.

But that is the trade being made: **~40 tokens of defensive prose on every single call, forever, to undo a temptation that only exists because the request is in scope.** The alternative is to not put the request in scope. Here is chapter 2's `Observe` as it was *first* written, before those sentences, given the same seconds-valued result — once with the whole conversation, once focused.

**4 of 4 against 0 of 4.** Fisher's exact test, one-sided: **p = 0.014**.

With the whole conversation in scope, chapter 2's original prompt read the rule — *"give every duration in milliseconds"* — and did as it was told: `1.15` became `1150 ms`, `2.9` became `2900 ms`. The arithmetic is right and no tool performed it. The numbers now in the conversation came from the model, and the agent's `calculate` tool, the one thing that could have been checked, was never called. That is chapter 2's `Observe` bug exactly, reproduced on demand.

Focused, the same prompt kept `2.9 s`, every time. Not because it was told not to convert — it was told nothing of the kind — but because **it could not see the request that wanted milliseconds.**

So there are two ways to stop a component doing something it shouldn't:

| | Cost | What it is |
|---|---|---|
| defensive prose in the prompt | ~40 tokens on **every** call, forever | telling the model not to be tempted |
| narrowing the context | one selection, cheaper than the prose | not tempting it |

Chapter 2 could only reach for the first, because it had nowhere to say the second. Both are worth having — the shipped `Observe` resisted 8 out of 8 with its guards *and* the rules in view — but only one of them scales to every new temptation somebody adds to a conversation later.

**This is the sharpest form of the chapter's claim.** Context is not a budget problem. What a component is shown decides what it will do.

## The other half of a policy: what the component does with what it gets

Narrowing a context is one of two things a policy decides. The other is whether the component should be running a model over this result **at all**.

`Observe` exists to turn a raw result into a sentence. For a JSON body that is a service. For a result that is *already* the artefact — a memory listing, a document fetched by id — it is damage, and `04_memory` measured both kinds: a listing lost the `#ids` that make entries openable in two live runs out of three, and an opened 868-character write-up came back at about 130 characters, with two of six specific facts surviving.

Chapter 2 already had the switch for this, `Observe(extractors=...)`, and chapter 4 ships the table: `EXTRACTORS` maps a tool name to "this output is final, hand it over untouched". No model call, nothing to paraphrase.

Here is `open_memory` coming back through the loop both ways:

**1 of 4 against 1 of 1, and the second one costs no model call at all.**

Three runs out of four, `Observe` wrote a perfectly good sentence about the memories and dropped every `#id` doing it. Nothing failed: the loop carries on, `Think` reads a summary of three memories it can no longer open, and `open_memory` is a tool it has no arguments for. The second stage is gone, and the only symptom is an answer written from summaries.

This is not the same kind of loss as the rest of the chapter. A dropped rule or a missing baseline makes an answer worse; **this removes a capability**, and it does it inside the component that was supposed to be the safe one.

Two things follow, and both belong to a policy rather than to a prompt:

- **`extractors` is part of what a component is given**, in the same sense a budget is. `Observe(extractors=EXTRACTORS)` says: for these tools the output *is* the artefact, hand it over. It is also the cheapest thing in this chapter — it removes a model call per tool result rather than adding one.
- **An id that matters must be in the content**, not in a label. Notebook 2's `retrieved()` now puts `#id` in the item's text for exactly this reason: `about` only survives if construction is asked for labels, and here even the text barely survived.

On a memory *body* rather than a listing the damage is milder and still real: measured on the runbook entry, the default path spent a call and returned 405 characters where the tool returned 401 — close, but not the same string, and not the string the store holds.

## A whole loop, one policy per component

The same session, run for real this time: two tools, a `Think` that decides, an `Act` that calls them, an `Observe` that phrases what came back. Each model-calling component gets its own context; `Act` gets none, because `Act` calls no model — there is nothing to give it.

A working agent: two tool calls, an answer that names the cause, the deploy and the numbers, and every model-calling component on its own policy. `Act` has none, because `Act` calls no model — the runtime's only component that needs no context at all.

The last line is what the loop declaration could never say before: **`Think` was given 544 tokens and `Observe` 33.** Unfocused, `Observe` got 851 for the same job.

And `02_agent_runtime` was not touched to do any of this. That was the real test of chapter 2's design: a component is anything with `.run(conversation)`, so something that narrows a conversation and forwards the call is also a component. Chapter 4 found the same thing about memory, where `Recall` slots in beside `Observe`. Two chapters now have needed a hook that already existed.

## What you built

`focus.py`:

- **`View`**: a conversation with different `messages` and the same everything else. Writes to anything but `messages` land on the real conversation, so `pending`, `step_outputs` and the budgets keep working.
- **`to_messages(items)`**: items back into the runtime's shapes, with the `decision` + `tool` pairing rebuilt so `Think` still produces native tool calls.
- **`Focus(component, **policy)`**: one component and the context it is given. `prepare(items, conversation)` is for what selection cannot express — a boundary, a compression, a retrieval — and it runs per call, so what a component gets is decided when it runs rather than when the loop is declared.
- **`current_pass`**: `Observe`'s policy, and a boundary rather than a score — the standing request and the decision in flight, and nothing else. It drops the pinned rules on purpose, which is what stops the unit conversion.
- **`sizes(focused)`**: what each component was last given, in tokens.

Two changes were needed outside `focus.py`, both in this chapter: `available()` now emits a `decision` item for a call nothing has reported on yet (without it, the component whose job is to describe that call could not see which call it was), and `select()` gained `keep_last=N`, because recency as a *weight* guarantees nothing — see notebook 2.

What this notebook measured, in one line: giving `Observe` less cost nothing (same values, 398 tokens against 851, and 33 in the live loop), and it prevented a failure chapter 2 could only prevent with prose — 4/4 conversions against 0/4, p = 0.014.

What is still missing is a place to say all of this at once. `thinking` and `watching` above are two policies written beside each other by hand, and the compression from notebook 3 and the retrieval from notebook 2 are not wired into either. Notebook 6 puts them together: select and retrieve running side by side, compression when something triggers it, construction at the end, and one policy per component.